# S&P 500 option-price Z-scores — Bloomberg BQNT only

Run all cells in an entitled Bloomberg BQuant notebook, then click **Load / refresh Bloomberg**. All market-data requests use `bql.Service()`. There is no synthetic-data mode, CSV input, or unconfigured premium-field callback.

**Methodology:** for every current S&P 500 constituent, select a current listed call/put pair near the chosen target expiry and closest strike to spot. Display each selected contract's latest available daily bid/ask midpoint and compare that exact contract with its own previous 365 calendar days of prices.

**This is not a constant-maturity / historically rolled ATM series.** Unlike the standardized-series proposal in the previous starter, this implementation uses currently selected listed contracts throughout their own histories. Maturity and moneyness change through that history. Recently listed contracts may have no full-year Z-score; missing scores remain blank.

The score excludes the current observation, requires at least 200 valid prior observations and 365 days of elapsed history, and uses sample standard deviation. About two years of daily history are requested to support rolling charts. The current universe is not a point-in-time index backtest.

**Validation:** the arithmetic, quote filters, contract selection and response joins were tested locally using independent test fixtures and a mocked transport. The proprietary Bloomberg requests and notebook widget rendering have not been tested in a live BQNT session. Field availability, historical bid/ask coverage, fill semantics and licensing must be verified there. This is a prototype, not a production-validated pricing monitor.


## 1. Settings

`MAX_STOCKS = None` requests every current constituent. Set it to `5` for an initial connection check. Target DTE and the allowed expiry band can also be changed in the dashboard before refreshing.

The only data path is Bloomberg BQL. `MODE = "BQNT"` is an informational label, not a Bloomberg API parameter. The BQL session is instantiated when the interface is launched.


In [ ]:
"""Bloomberg BQNT-only S&P 500 listed-option price Z-score dashboard.

No synthetic-data mode, CSV loader, or unconfigured premium-field callbacks.
Run in an entitled Bloomberg BQuant notebook. Live BQL execution is untested here.

Method: select a CURRENT listed call/put pair near the target expiry and ATM
strike; compare each exact contract's premium to its OWN prior 365-day history.
This is NOT a historical constant-maturity / constant-moneyness option series.
"""
from __future__ import annotations

from functools import reduce
from html import escape
from typing import Callable, Optional

import numpy as np
import pandas as pd

MODE = "BQNT"  # Informational only: Bloomberg is the only data path.
INDEX = "SPX Index"
TARGET_DTE = 30
EXPIRY_TOLERANCE_DAYS = 14
MIN_DTE = 7
WINDOW_DAYS = 365
MIN_OBSERVATIONS = 200
BATCH_SIZE = 40
MAX_STOCKS = None  # None = every current constituent; 5 = connection test.

## 2. Validation and rolling calculations

In [ ]:
REQUIRED = ["date", "ticker", "spot", "call_mid", "put_mid"]
METRICS = {"Call Z": "call", "Put Z": "put", "Spread Z": "spread"}


def validate_history(raw: pd.DataFrame) -> pd.DataFrame:
    """Validate daily observations for each currently selected listed-option pair.

    Prices must be USD per underlying share; dates must be timezone-naive US
    market-date labels. Missing observations are allowed but never filled.
    The caller must verify fixed contract identifiers and price conventions;
    the five-column schema alone cannot establish these properties.
    """
    if not isinstance(raw, pd.DataFrame) or raw.empty:
        raise ValueError("The history loader must return a nonempty DataFrame.")
    missing = set(REQUIRED).difference(raw.columns)
    if missing:
        raise ValueError("Missing input columns: " + ", ".join(sorted(missing)))
    d = raw[REQUIRED].copy()
    d["date"] = pd.to_datetime(d["date"], errors="raise")
    if d["date"].isna().any():
        raise ValueError("Dates cannot be missing.")
    if d["date"].dt.tz is not None:
        raise ValueError("Use timezone-naive US market-date labels, not timestamps.")
    d["date"] = d["date"].dt.normalize()
    if d["ticker"].isna().any():
        raise ValueError("Ticker values cannot be missing.")
    d["ticker"] = d["ticker"].astype(str).str.strip()
    if d["ticker"].eq("").any():
        raise ValueError("Ticker values cannot be blank.")
    if d.duplicated(["date", "ticker"]).any():
        raise ValueError("Duplicate date/ticker rows: select a single consistent option series first.")
    for col in ["spot", "call_mid", "put_mid"]:
        d[col] = pd.to_numeric(d[col], errors="raise").astype(float)
        if np.isinf(d[col]).any():
            raise ValueError(f"Infinite values in {col}.")
        invalid = d[col].le(0) if col == "spot" else d[col].lt(0)
        if invalid.any():
            raise ValueError(f"Invalid negative/zero price in {col}.")
    return d.sort_values(["date", "ticker"]).reset_index(drop=True)


def compute_dashboard(
    raw: pd.DataFrame,
    basis: str = "Raw premium",
    window_days: int = 365,
    min_obs: int = 200,
    asof: Optional[str] = None,
) -> dict:
    """Calculate per-stock trailing z-scores, never cross-sectional z-scores.

    Baseline at t: dates in [t-window_days, t); today's value is excluded.
    A score requires min_obs nonmissing prior observations AND at least
    window_days of elapsed history for that stock/metric. Standard deviation
    uses ddof=1. A zero or invalid standard deviation produces NaN, not zero.
    This is a 365-day convention, not an exact anniversary in leap years.
    """
    if window_days < 2 or min_obs < 2:
        raise ValueError("window_days and min_obs must both be >= 2.")
    if basis not in {"Raw premium", "Premium / spot"}:
        raise ValueError("Unknown price basis.")
    d = validate_history(raw)
    stamp = pd.Timestamp(asof).normalize() if asof else d["date"].max()
    if stamp.tzinfo is not None:
        raise ValueError("asof must be a timezone-naive market date.")
    d = d.loc[d["date"].le(stamp)].copy()
    if d.empty:
        raise ValueError("There are no observations on or before the requested date.")
    tickers = sorted(d["ticker"].unique())
    dates = pd.DatetimeIndex(sorted(set(d["date"]) | {stamp}), name="date")
    price = {
        col: d.pivot(index="date", columns="ticker", values=col).reindex(
            index=dates, columns=tickers
        )
        for col in ["spot", "call_mid", "put_mid"]
    }
    call, put = price["call_mid"], price["put_mid"]
    if basis == "Premium / spot":
        call = 100.0 * call.div(price["spot"])
        put = 100.0 * put.div(price["spot"])
    values = {"call": call, "put": put, "spread": call - put}
    scores, counts, means, stds, statuses = {}, {}, {}, {}, {}
    delta = pd.Timedelta(days=window_days)
    for metric, x in values.items():
        rolling = x.rolling(f"{window_days}D", closed="left", min_periods=2)
        mean, std = rolling.mean(), rolling.std(ddof=1)
        count = x.rolling(f"{window_days}D", closed="left", min_periods=0).count()
        mature = pd.DataFrame(False, index=dates, columns=tickers)
        for ticker in tickers:
            first = x[ticker].first_valid_index()
            if first is not None:
                mature[ticker] = dates >= first + delta
        eligible = count.ge(min_obs) & mature
        z = ((x - mean) / std.where(std.gt(1e-12))).where(eligible)
        scores[metric], counts[metric] = z, count
        means[metric], stds[metric] = mean, std
        status = pd.Series("OK", index=tickers, dtype=object)
        status.loc[~std.loc[stamp].gt(1e-12)] = "Zero/invalid standard deviation"
        status.loc[count.loc[stamp].lt(min_obs)] = f"Fewer than {min_obs} prior observations"
        status.loc[~mature.loc[stamp]] = f"Less than {window_days} days of history"
        status.loc[x.loc[stamp].isna()] = "No value at the as-of date"
        statuses[metric] = status

    latest = pd.DataFrame(index=pd.Index(tickers, name="Ticker"))
    latest["Spot"] = price["spot"].loc[stamp]
    latest["Call mid"] = price["call_mid"].loc[stamp]
    latest["Put mid"] = price["put_mid"].loc[stamp]
    for label, metric in METRICS.items():
        latest[label] = scores[metric].loc[stamp]
    quote_rows = d.loc[d[["call_mid", "put_mid"]].notna().any(axis=1)]
    last_quote = quote_rows.groupby("ticker")["date"].max()
    latest["Last option date"] = last_quote.reindex(tickers)
    return dict(
        latest=latest, scores=scores, values=values, price=price,
        counts=counts, means=means, stds=stds, statuses=statuses,
        asof=stamp, window_days=window_days, min_obs=min_obs, basis=basis,
    )

## 3. Direct BQL data retrieval and contract selection

In [ ]:
def request_frame(bq, universe, items: dict, history: bool = False) -> pd.DataFrame:
    """Join named BQL results explicitly; reject ambiguous observations."""
    import bql
    response = bq.execute(bql.Request(universe, items))
    if len(response) != len(items):
        raise ValueError("BQL returned a different number of fields than requested.")
    keys = ["ID", "DATE"] if history else ["ID"]
    pieces = []
    for name, result in zip(items, response):
        frame = result.df().reset_index()
        if frame.empty:
            pieces.append(pd.DataFrame(columns=keys + [name]))
            continue
        missing = set(keys + [name]).difference(frame.columns)
        if missing:
            raise ValueError(f"{name}: missing BQL columns {sorted(missing)}; got {list(frame.columns)}")
        if "CURRENCY" in frame and history:
            currencies = set(frame["CURRENCY"].dropna().astype(str).str.upper())
            if currencies - {"USD"}:
                raise ValueError(f"{name}: expected USD prices; got {currencies}.")
        frame = frame[keys + [name]].copy()
        if frame["ID"].isna().any():
            raise ValueError(f"{name}: BQL returned a missing security ID.")
        frame["ID"] = frame["ID"].astype(str)
        if history:
            frame["DATE"] = pd.to_datetime(frame["DATE"], errors="raise")
            if frame["DATE"].isna().any() or frame["DATE"].dt.tz is not None:
                raise ValueError(f"{name}: expected timezone-naive market dates.")
            frame["DATE"] = frame["DATE"].dt.normalize()
            frame[name] = pd.to_numeric(frame[name], errors="raise").replace([np.inf, -np.inf], np.nan)
        if frame.duplicated(keys).any():
            raise ValueError(f"{name}: duplicate {keys}; inspect returned BQL dimensions.")
        pieces.append(frame)
    return reduce(
        lambda a, b: a.merge(b, on=keys, how="outer", validate="one_to_one"),
        pieces,
    )


def select_pair(chain: pd.DataFrame, spot: float, asof: pd.Timestamp,
                target_dte: int, tolerance: int) -> dict:
    """Closest eligible expiry first, then closest strike with one call and put.

    Reject duplicate expiry/strike/side records rather than silently choosing an
    exchange or adjusted contract. Unique pairs still require deliverable review.
    """
    if not np.isfinite(spot) or spot <= 0:
        raise ValueError("No valid underlying price on the common as-of date.")
    c = chain.copy()
    c["Expiry"] = pd.to_datetime(c["Expiry"], errors="coerce").dt.normalize()
    c["Strike"] = pd.to_numeric(c["Strike"], errors="coerce")
    c["Side"] = c["Side"].astype(str).str.strip().str.upper().map(
        {"CALL": "C", "C": "C", "PUT": "P", "P": "P"}
    )
    c["DTE"] = (c["Expiry"] - asof).dt.days
    c = c.loc[c["Side"].notna() & c["Strike"].gt(0)
              & c["DTE"].ge(MIN_DTE)
              & c["DTE"].sub(target_dte).abs().le(tolerance)].copy()
    c = c.loc[~c.duplicated(["Expiry", "Strike", "Side"], keep=False)]
    if c.empty:
        raise ValueError("No unambiguous option pair within the requested expiry band.")
    pairs = c.pivot(index=["Expiry", "Strike", "DTE"], columns="Side", values="ID")
    if not {"C", "P"}.issubset(pairs.columns):
        raise ValueError("No matched call and put in the eligible chain.")
    pairs = pairs.dropna(subset=["C", "P"]).reset_index()
    if pairs.empty:
        raise ValueError("No call and put share an eligible expiry and strike.")
    pairs["expiry_distance"] = pairs["DTE"].sub(target_dte).abs()
    pairs["strike_distance"] = pairs["Strike"].sub(spot).abs()
    p = pairs.sort_values(["expiry_distance", "Expiry", "strike_distance", "Strike"]).iloc[0]
    return {"Expiry": p["Expiry"], "Strike": float(p["Strike"]),
            "DTE": int(p["DTE"]), "Call contract": p["C"], "Put contract": p["P"]}


def quote_midpoints(quotes: pd.DataFrame) -> pd.DataFrame:
    """Midpoints only where both quotes exist and ask >= bid >= 0, ask > 0."""
    q = quotes.copy()
    valid = (np.isfinite(q["Bid"]) & np.isfinite(q["Ask"])
             & q["Bid"].ge(0) & q["Ask"].gt(0) & q["Ask"].ge(q["Bid"]))
    q["Mid"] = ((q["Bid"] + q["Ask"]) / 2.0).where(valid)
    q["Zero bid"] = valid & q["Bid"].eq(0)
    return q


def fetch_bloomberg_history(bq, progress: Callable[[str], None] = lambda _: None,
                           target_dte: int = TARGET_DTE,
                           tolerance: int = EXPIRY_TOLERANCE_DAYS,
                           max_stocks: Optional[int] = MAX_STOCKS) -> dict:
    """Fetch current SPX members, selected listed contracts, and daily prices.

    The '0D' endpoint requests the latest available daily observation; this is
    not a streaming subscription or a guarantee of an up-to-the-second quote.
    No forward filling, theoretical-pricing fallback, or replacement data source.
    Confirm BQL fields, historical bid/ask coverage, and entitlements in BQNT.
    """
    if target_dte < MIN_DTE or tolerance < 0:
        raise ValueError(f"Target DTE must be >= {MIN_DTE}; tolerance must be >= 0.")
    if max_stocks is not None and max_stocks < 1:
        raise ValueError("MAX_STOCKS must be None or a positive integer.")
    progress("Retrieving current SPX Index members...")
    members = request_frame(bq, bq.univ.members(INDEX), {"Name": bq.data.name()})
    if members.empty:
        raise ValueError("Bloomberg returned no current index constituents.")
    members = members.sort_values("ID").set_index("ID")
    if max_stocks is not None:
        members = members.head(max_stocks)
    tickers = members.index.tolist()

    # Use date-stamped underlying history to establish one common market date.
    dates = bq.func.range("-2Y", "0D")
    spot_parts = []
    for i in range(0, len(tickers), BATCH_SIZE):
        progress(f"Underlying history: {min(i+BATCH_SIZE, len(tickers))}/{len(tickers)} stocks")
        spot_parts.append(request_frame(
            bq, tickers[i:i+BATCH_SIZE],
            {"spot": bq.data.px_last(dates=dates, fill="NA")}, history=True,
        ))
    spots = pd.concat(spot_parts, ignore_index=True)
    spots = spots.loc[spots["spot"].gt(0)].copy()
    if spots.empty:
        raise ValueError("No valid underlying history was returned.")
    asof = pd.Timestamp(spots["DATE"].max())
    current_spot = spots.loc[spots["DATE"].eq(asof)].set_index("ID")["spot"]
    meta = members.rename_axis("Ticker").copy()
    meta["Selection status"] = "Not processed"
    errors = []
    pairs = []
    chain_items = {"Expiry": bq.data.expire_dt(),
                   "Strike": bq.data.strike_px(), "Side": bq.data.put_call()}
    completed_chain_requests = 0
    failed_chain_requests = 0
    for i, ticker in enumerate(tickers, 1):
        progress(f"Selecting listed call/put pairs: {i}/{len(tickers)} — {ticker}")
        try:
            if ticker not in current_spot:
                raise ValueError("No underlying price on the common as-of date.")
            try:
                chain = request_frame(bq, bq.univ.options(ticker), chain_items)
            except Exception:
                failed_chain_requests += 1
                raise
            completed_chain_requests += 1
            selected = select_pair(chain, float(current_spot[ticker]), asof, target_dte, tolerance)
            pairs.append({"Ticker": ticker, **selected})
            meta.loc[ticker, "Selection status"] = "OK"
        except Exception as exc:
            message = str(exc)
            meta.loc[ticker, "Selection status"] = message
            errors.append({"Stage": "Option selection", "Security": ticker, "Error": message})
            # Avoid hundreds of identical queries after an API/schema failure.
            if completed_chain_requests == 0 and failed_chain_requests >= 3:
                raise RuntimeError("The first option-chain requests failed. " + message) from exc
    if pairs:
        selected = pd.DataFrame(pairs).set_index("Ticker")
        meta = meta.join(selected)
    else:
        for col in ["Expiry", "Strike", "DTE", "Call contract", "Put contract"]:
            meta[col] = np.nan

    option_ids = sorted(set(meta["Call contract"].dropna()) | set(meta["Put contract"].dropna()))
    history_dates = bq.func.range("-2Y", asof.strftime("%Y-%m-%d"))
    # Concrete BQL data items: these are listed-option bid/ask prices, not IV.
    quote_items = {"Bid": bq.data.px_bid(dates=history_dates, fill="NA"),
                   "Ask": bq.data.px_ask(dates=history_dates, fill="NA")}
    quote_parts = []
    failed_option_ids = set()
    for i in range(0, len(option_ids), BATCH_SIZE):
        batch = option_ids[i:i+BATCH_SIZE]
        progress(f"Option bid/ask history: {min(i+BATCH_SIZE, len(option_ids))}/{len(option_ids)} contracts")
        try:
            quote_parts.append(request_frame(bq, batch, quote_items, history=True))
        except Exception as exc:
            if i == 0:
                raise RuntimeError(
                    "First historical PX_BID/PX_ASK request failed; check these "
                    "items and their history entitlement in BQNT. " + str(exc)
                ) from exc
            failed_option_ids.update(batch)
            errors.append({"Stage": "Option history", "Security": ", ".join(batch), "Error": str(exc)})
    quotes = (quote_midpoints(pd.concat(quote_parts, ignore_index=True))
              if quote_parts else pd.DataFrame(columns=["ID", "DATE", "Bid", "Ask", "Mid", "Zero bid"]))
    by_id = {key: group for key, group in quotes.groupby("ID")}
    by_stock = {key: group for key, group in spots.groupby("ID")}
    rows = []
    for ticker in tickers:
        spot = by_stock.get(ticker, pd.DataFrame(columns=["DATE", "spot"]))
        panel = spot[["DATE", "spot"]].set_index("DATE")
        for side, col in [("Call", "call_mid"), ("Put", "put_mid")]:
            option_id = meta.loc[ticker, f"{side} contract"]
            q = by_id.get(option_id, pd.DataFrame(columns=["DATE", "Mid", "Zero bid"]))
            series = q.set_index("DATE")["Mid"].rename(col)
            panel = panel.join(series, how="outer", validate="one_to_one")
            valid_dates = q.loc[q["Mid"].notna(), "DATE"]
            meta.loc[ticker, f"{side} last date"] = valid_dates.max() if not valid_dates.empty else pd.NaT
            on_date = q.loc[q["DATE"].eq(asof)]
            meta.loc[ticker, f"{side} zero bid"] = bool(on_date["Zero bid"].any())
            if option_id in failed_option_ids:
                meta.loc[ticker, "Selection status"] = "Option-history request failed; see query errors"
        panel = panel.reindex(panel.index.union(pd.DatetimeIndex([asof])))
        panel.index.name = "date"
        panel["ticker"] = ticker
        for col in ["spot", "call_mid", "put_mid"]:
            panel[col] = pd.to_numeric(panel[col], errors="raise").astype(float)
        rows.append(panel.reset_index())
    raw = validate_history(pd.concat(rows, ignore_index=True))
    return {"history": raw, "contracts": meta, "errors": pd.DataFrame(errors),
            "asof": asof, "target_dte": target_dte, "tolerance": tolerance}

## 4. BQNT dashboard widgets

In [ ]:
def launch_dashboard(bq=None):
    """Create notebook widgets. The Load button is the only data-loading trigger."""
    import bql
    import ipywidgets as w
    import matplotlib.pyplot as plt
    from IPython.display import HTML, display
    bq = bq if bq is not None else bql.Service()
    target = w.IntText(value=TARGET_DTE, description="Target DTE:")
    band = w.IntText(value=EXPIRY_TOLERANCE_DAYS, description="DTE band ±:")
    basis = w.Dropdown(options=["Raw premium", "Premium / spot"], description="Basis:")
    metric = w.Dropdown(options=["Call Z", "Put Z", "Spread Z"], value="Call Z", description="Rank by:")
    minimum = w.FloatSlider(value=0, min=0, max=4, step=0.5, description="Min |Z|:", continuous_update=False)
    top_n = w.Dropdown(options=[20, 50, 100, 600], value=50, description="Max rows:")
    search = w.Text(placeholder="Ticker contains...", description="Search:", continuous_update=False)
    stock = w.Dropdown(options=[], description="Chart:", layout=w.Layout(width="330px"))
    refresh = w.Button(description="Load / refresh Bloomberg", icon="refresh", layout=w.Layout(width="240px"))
    progress = w.HTML()
    banner = w.HTML(value=(
        "<h3>S&amp;P 500 listed-option price Z-scores | BQNT</h3>"
        "<p>Click <b>Load / refresh Bloomberg</b>. No demo or CSV data path.<br>"
        "Each selected contract is compared with its own history. This is NOT a "
        "constant-maturity or historically rolled ATM series.</p>"
    ))
    table_out, chart_out, error_out = w.Output(), w.Output(), w.Output()
    state = {"data": None, "models": {}, "loading": False}

    def model():
        if basis.value not in state["models"]:
            state["models"][basis.value] = compute_dashboard(
                state["data"]["history"], basis.value, WINDOW_DAYS, MIN_OBSERVATIONS,
                asof=state["data"]["asof"],
            )
        return state["models"][basis.value]

    def draw_table(*_):
        if state["loading"] or state["data"] is None:
            return
        m = model()
        key = METRICS[metric.value]
        all_rows = m["latest"].join(state["data"]["contracts"])
        all_rows["Prior observations"] = m["counts"][key].loc[m["asof"]].astype(int)
        all_rows["Z status"] = m["statuses"][key]
        bad_selection = all_rows["Selection status"].ne("OK")
        all_rows.loc[bad_selection, "Z status"] = all_rows.loc[bad_selection, "Selection status"]
        t = all_rows.copy()
        if search.value.strip():
            t = t.loc[t.index.str.contains(search.value.strip(), case=False, regex=False)]
        if minimum.value > 0:
            t = t.loc[t[metric.value].abs().ge(minimum.value)]
        t = t.assign(_rank=t[metric.value].abs()).sort_values("_rank", ascending=False, na_position="last").head(top_n.value)
        columns = ["Name", "Spot", "Expiry", "DTE", "Strike", "Call mid", "Put mid",
                   "Call Z", "Put Z", "Spread Z", "Prior observations", "Z status",
                   "Call last date", "Put last date", "Call zero bid", "Put zero bid",
                   "Call contract", "Put contract"]
        t = t.reindex(columns=columns)
        for col in ["Expiry", "Call last date", "Put last date"]:
            t[col] = pd.to_datetime(t[col]).dt.strftime("%Y-%m-%d").fillna("—")
        with table_out:
            table_out.clear_output(wait=True)
            display(HTML(f"<p><b>{all_rows[metric.value].notna().sum()}/{len(all_rows)} stocks have a valid {escape(metric.value)}.</b> "
                         "Missing latest prices are not carried forward. Z-score ranking is by absolute value.</p>"))
            display(HTML('<div style="overflow-x:auto;max-height:520px;overflow-y:auto">'
                         + t.to_html(float_format=lambda v: f"{v:,.2f}", na_rep="—", escape=True) + "</div>"))

    def draw_chart(*_):
        if state["loading"] or state["data"] is None or not stock.value:
            return
        m, ticker = model(), stock.value
        left = m["asof"] - pd.Timedelta(days=365)
        info = state["data"]["contracts"].loc[[ticker]]
        with chart_out:
            chart_out.clear_output(wait=True)
            display(HTML(info[["Call contract", "Put contract", "Selection status"]].to_html(escape=True)))
            fig, ax = plt.subplots(figsize=(11, 3.2))
            for label, key in [("Call Z", "call"), ("Put Z", "put")]:
                z = m["scores"][key].loc[left:, ticker]
                ax.plot(z.index, z.values, label=label)
            ax.axhline(2, linestyle="--", linewidth=0.8)
            ax.axhline(-2, linestyle="--", linewidth=0.8)
            ax.axhline(0, linestyle=":", linewidth=0.8)
            ax.set_title(f"{ticker} | Same-contract rolling 365-day Z | {basis.value}")
            ax.set_ylabel("Standard deviations")
            ax.legend()
            fig.tight_layout()
            plt.show()
            plt.close(fig)
            fig, ax = plt.subplots(figsize=(11, 3.0))
            for label, key in [("Call mid", "call_mid"), ("Put mid", "put_mid")]:
                p = m["price"][key].loc[left:, ticker]
                ax.plot(p.index, p.values, label=label)
            ax.set_title(f"{ticker} | Currently selected contracts' daily premiums")
            ax.set_ylabel("USD per underlying share")
            ax.legend()
            fig.tight_layout()
            plt.show()
            plt.close(fig)

    def refresh_data(*_):
        state["loading"] = True
        state["data"], state["models"] = None, {}
        refresh.disabled = target.disabled = band.disabled = True
        table_out.clear_output()
        chart_out.clear_output()
        error_out.clear_output()
        try:
            data = fetch_bloomberg_history(
                bq, progress=lambda text: setattr(progress, "value", escape(text)),
                target_dte=int(target.value), tolerance=int(band.value), max_stocks=MAX_STOCKS,
            )
            state["data"] = data
            tickers = sorted(data["history"]["ticker"].unique())
            old = stock.value
            stock.options = tickers
            stock.value = old if old in tickers else tickers[0]
            loaded = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC")
            banner.value = (
                "<h3>S&amp;P 500 listed-option price Z-scores | BQNT</h3>"
                f"<p><b>Bloomberg BQL data | Common as-of date: {data['asof']:%Y-%m-%d}</b><br>"
                f"Loaded {loaded}. Latest available daily observations; NOT a streaming feed.<br>"
                f"Current listed pairs: target {data['target_dte']} DTE ± {data['tolerance']} days, "
                "closest eligible expiry then closest strike to spot; identical expiry/strike for call and put.<br>"
                f"Baseline: previous {WINDOW_DAYS} calendar days, excluding the current date; "
                f"at least {MIN_OBSERVATIONS} valid observations and a full elapsed year.<br>"
                "<b>Same-contract history, NOT constant-maturity history.</b> Recently listed contracts can have no valid Z-score. "
                "Check quote timing, spreads, deliverables and corporate actions before relying on this prototype.</p>"
            )
            progress.value = f"Loaded {len(tickers)} current constituent rows; {len(data['errors'])} query/selection issues."
            if not data["errors"].empty:
                with error_out:
                    display(HTML(data["errors"].to_html(index=False, escape=True)))
        except Exception as exc:
            banner.value = "<h3>Bloomberg loading failed</h3><p>" + escape(str(exc)) + "</p>"
            progress.value = "No substitute data was loaded."
        finally:
            state["loading"] = False
            refresh.disabled = target.disabled = band.disabled = False
        if state["data"] is not None:
            try:
                draw_table()
                draw_chart()
            except Exception as exc:
                with error_out:
                    display(HTML("<b>Rendering/calculation error:</b> " + escape(str(exc))))

    for control in [metric, minimum, top_n, search]:
        control.observe(draw_table, names="value")
    stock.observe(draw_chart, names="value")
    basis.observe(lambda *_: (draw_table(), draw_chart()), names="value")
    refresh.on_click(refresh_data)
    errors_panel = w.Accordion(children=[error_out])
    errors_panel.set_title(0, "Query / selection issues")
    errors_panel.selected_index = None
    controls = w.Box([basis, metric, minimum, top_n, search], layout=w.Layout(display="flex", flex_flow="row wrap"))
    return w.VBox([banner, w.HBox([target, band, refresh]), progress, controls,
                   table_out, stock, chart_out, errors_panel])

## 5. Launch

Run this cell, then click **Load / refresh Bloomberg**.

In [ ]:
import bql
from IPython.display import display

bq = bql.Service()
dashboard = launch_dashboard(bq)
display(dashboard)

## Interpretation and data checks

Prices are daily bid/ask midpoints, not implied volatilities and not theoretical model prices. The application rejects missing, negative or crossed quotes; zero bids with positive asks are retained and flagged. No missing observations are forward-filled. The displayed as-of date is the latest date returned in the underlying history; it is not proof that every exchange quote was refreshed at that instant.

Call and put selection requires identical expiry and strike and rejects duplicate expiry/strike/side rows. This does **not** by itself verify identical deliverables or standard contract adjustments. Review corporate actions, adjusted contracts, quote timestamps, liquidity and the chosen pricing source before relying on the output. Refresh can select a different current contract; the historical chart then refers to that newly selected contract throughout.

The ranking is by absolute Z-score. A large score means an unusual premium relative to the specified historical baseline, not a buy/sell recommendation. Spread Z is the Z-score of call-minus-put premiums, not the difference between the two separate Z-scores.

Query and selection failures are exposed in the expandable error panel. Unsupported history requests do not fall back to invented data or implied volatility.

### Reference material

- Bloomberg BQuant platform: https://professional.bloomberg.com/products/bloomberg-terminal/research/bquant/
- pandas date-based rolling windows: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rolling.html
- Published first-hand BQL option-chain example, including `options`, `expire_dt`, `strike_px` and `put_call`: https://medium.com/@annievelcu/visualising-the-effects-of-the-russo-ukrainian-war-in-the-markets-case-study-with-bql-and-bquant-bc908e782206

These public sources do not verify your live BQL version or historical PX_BID/PX_ASK entitlement. Use the field documentation in your Bloomberg session to resolve any rejected query.
